# Module 06: Extra Model - Random Forest with SMOTE

Advanced ensemble learning combined with Synthetic Minority Over-sampling Technique (SMOTE).

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Loading the data
try:
    df_credit = pd.read_csv("../creditcard.csv")
    print("Loaded ../creditcard.csv successfully.")
except FileNotFoundError:
    df_credit = pd.read_csv("creditcard.csv")
    print("Loaded creditcard.csv successfully.")

In [ ]:
# Top rows
df_credit.head()

In [ ]:
# Grid of plots for key features distribution comparing Normal vs Fraud
features = ['V1', 'V2', 'V3', 'V4', 'V9', 'V10', 'V11', 'V12', 'V14', 'V16', 'V17', 'V18']

plt.figure(figsize=(14, 16))
gs = gridspec.GridSpec(4, 3)

for i, col in enumerate(features):
    ax = plt.subplot(gs[i])
    sns.kdeplot(df_credit[df_credit['Class'] == 0][col], color='blue', label='Normal', ax=ax)
    sns.kdeplot(df_credit[df_credit['Class'] == 1][col], color='red', label='Fraud', ax=ax)
    ax.set_title(f'Distribution of {col}', fontsize=11, fontweight='bold')
    if i == 0:
        ax.legend()

plt.tight_layout()
plt.show()

## Train-Test Split and Preprocessing

In [ ]:
# Preprocessing
scaler = StandardScaler()
df_rf = df_credit.copy()
df_rf['Amount_Scaled'] = scaler.fit_transform(df_rf[['Amount']])
df_rf.drop(columns=['Time', 'Amount'], inplace=True)

X = df_rf.drop(columns=['Class'])
y = df_rf['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Before SMOTE - Normal (0): {(y_train == 0).sum():,}, Fraud (1): {(y_train == 1).sum():,}")

## Applying SMOTE

In [ ]:
# Apply SMOTE only to training data
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f"After SMOTE - Normal (0): {(y_train_res == 0).sum():,}, Fraud (1): {(y_train_res == 1).sum():,}")
print("Classes are balanced 50:50 in training split.")

## Training Random Forest Classifier

In [ ]:
# Train Random Forest on SMOTE-augmented dataset
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_res, y_train_res)

# Predictions on untouched real test set
y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

print("Random Forest + SMOTE Results:")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_rf):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_rf):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_rf):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob_rf):.4f}")

In [ ]:
# Classification Report
print("Classification Report:\n")
print(classification_report(y_test, y_pred_rf, target_names=['Normal', 'Fraud'], digits=4))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_rf)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', cbar=False)
plt.title(f'Random Forest + SMOTE Confusion Matrix\nRecall: {recall_score(y_test, y_pred_rf):.2%}, Precision: {precision_score(y_test, y_pred_rf):.2%}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks([0.5, 1.5], ['Normal', 'Fraud'])
plt.yticks([0.5, 1.5], ['Normal', 'Fraud'])
plt.show()

## Feature Importances in Random Forest

In [ ]:
# Top 15 Feature Importances
importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=importances.head(15).values, y=importances.head(15).index, palette='crest')
plt.title('Top 15 Features by Importance (Random Forest)', fontsize=13, fontweight='bold')
plt.xlabel('Mean Decrease in Impurity')
plt.show()